# Pizza Palace Wetteren Sales Optimization

In [2]:
import pandas as pd
import glob

### Step 1: Load and merge the data

In [21]:
# 1. Get all CSV file paths in the 'Order' folder
csv_files = glob.glob("Order/Order_*.csv")

# 2. Load each file and combine them into a single DataFrame
df_list = [pd.read_csv(file) for file in csv_files]
df = pd.concat(df_list, ignore_index=True)

# Display a preview of the data and available columns
print("Data successfully loaded!")
df.head()

Data successfully loaded!


,Order,Date,Postcode,Total amount,Paid online,Pickup
0,R8WC83,4/1/2026 10:42,NaN,5.5,yes,yes
1,V87YQH,4/1/2026 17:45,NaN,24.5,yes,yes
2,MTRJQD,4/1/2026 17:49,9230.0,17.0,no,no
3,MQHVXH,4/1/2026 17:53,9520.0,30.1,yes,no
4,P77GVF,4/1/2026 18:06,9230.0,39.0,yes,no


## Business Questions To Solve & Optimize 📊 

### De Maandelijkse Conjunctuur (Jan-Jun)

In [18]:
# 1. Maandelijkse Omzet, Orders en AOV
df['Month'] = df['Date'].dt.to_period('M')

monthly_stats = df.groupby('Month').agg(
    Total_Revenue=('Total amount', 'sum'),
    Order_Count=('Order', 'count'),
    Average_Order_Value=('Total amount', 'mean')
).reset_index()

print("--- MAANDELIJKSE BUSINESS PRESTATIES ---")
print(monthly_stats.to_string(index=False))

# Tip voor diepgang: Bereken de procentuele verandering t.o.v. de vorige maand
monthly_stats['Revenue_Growth_%'] = monthly_stats['Total_Revenue'].pct_change() * 100
print("\n--- MAAND-OP-MAAND GROEI ---")
print(monthly_stats[['Month', 'Revenue_Growth_%']].to_string(index=False))

--- MAANDELIJKSE BUSINESS PRESTATIES ---
  Month  Total_Revenue  Order_Count  Average_Order_Value
2026-01       12505.20          489            25.573006
2026-02       10025.80          380            26.383684
2026-03       10131.30          394            25.713959
2026-04       10993.00          417            26.362110
2026-05       12242.15          460            26.613370
2026-06       12369.45          452            27.366040

--- MAAND-OP-MAAND GROEI ---
  Month  Revenue_Growth_%
2026-01               NaN
2026-02        -19.826952
2026-03          1.052285
2026-04          8.505325
2026-05         11.363140
2026-06          1.039850


### Het Logistieke Duel (Afhaal vs. Bezorging)

In [22]:
# 1. Algemene splitsing Afhaal vs Bezorging
delivery_vs_pickup = df.groupby('Pickup').agg(
    Total_Revenue=('Total amount', 'sum'),
    Order_Count=('Order', 'count'),
    Average_Order_Value=('Total amount', 'mean')
).reset_index()

# Bereken het omzetaandeel (%)
total_biz_revenue = df['Total amount'].sum()
delivery_vs_pickup['Revenue_%'] = (delivery_vs_pickup['Total_Revenue'] / total_biz_revenue) * 100

print("--- LOGISTIEKE IMPACT: AFHAAL VS BEZORGING ---")
print(delivery_vs_pickup.to_string(index=False))

# 2. Diepere laag: Is er een geografisch patroon in afhalen?
geo_logistics = pd.crosstab(df['Postcode'], df['Pickup'], normalize='index') * 100
print("\n--- AFHAAL % PER POSTCODE (9230 vs de rest) ---")
print(geo_logistics)

--- LOGISTIEKE IMPACT: AFHAAL VS BEZORGING ---
Pickup  Total_Revenue  Order_Count  Average_Order_Value  Revenue_%
    no        65738.1         2440            26.941844  96.295716
   yes         2528.8          152            16.636842   3.704284

--- AFHAAL % PER POSTCODE (9230 vs de rest) ---
Pickup       no
Postcode       
9070.0    100.0
9090.0    100.0
9200.0    100.0
9230.0    100.0
9260.0    100.0
9270.0    100.0
9290.0    100.0
9340.0    100.0
9420.0    100.0
9520.0    100.0
9521.0    100.0
9550.0    100.0
9551.0    100.0
9552.0    100.0
9860.0    100.0


### Winkelmand-Gedrag (Micro-trends & Combinaties)

In [20]:
# Aantal pizza's en desserts schatten op basis van gemiddelde prijzen  (geen productdata)
# Stel: gemiddelde pizza = €12, gemiddeld dessert = €6
df['Estimated_Pizzas'] = (df['Total amount'] // 12).astype(int)

print("--- VERDELING AANTAL PIZZA'S PER BESTELLING ---")
print(df['Estimated_Pizzas'].value_counts(normalize=True) * 100)

--- VERDELING AANTAL PIZZA'S PER BESTELLING ---
Estimated_Pizzas
1     51.041667
2     29.591049
3     12.307099
4      2.777778
0      2.237654
5      1.273148
6      0.501543
7      0.192901
8      0.038580
11     0.038580
Name: proportion, dtype: float64


### Analyse op basis van het orderbedrag (Winstgevende drempels bepalen)

In [23]:

def cluster_order(amount):
    if amount < 15: return 'Kleine order (1 persoon / Snack)'
    elif amount <= 30: return 'Medium order (2 personen / 2 Pizza\'s)'
    elif amount <= 45: return 'Grote order (Gezin / 3 Pizza\'s)'
    else: return 'Groepsbestelling (4+ Pizza\'s)'

df['Order_Category'] = df['Total amount'].apply(cluster_order)

order_distribution = df.groupby('Order_Category').agg(
    Aantal_Orders=('Order', 'count'),
    Omzet_Bijdrage=('Total amount', 'sum')
).reset_index()

order_distribution['Omzet_%'] = (order_distribution['Omzet_Bijdrage'] / total_biz_revenue) * 100
print("--- WINKELMAND FORMATEN & HUN BUDGETTAIRE IMPACT ---")
print(order_distribution.sort_values(by='Omzet_%', ascending=False).to_string(index=False))

--- WINKELMAND FORMATEN & HUN BUDGETTAIRE IMPACT ---
                       Order_Category  Aantal_Orders  Omzet_Bijdrage   Omzet_%
Medium order (2 personen / 2 Pizza's)           1733         36114.7 52.902212
      Grote order (Gezin / 3 Pizza's)            605         21848.7 32.004822
        Groepsbestelling (4+ Pizza's)            164          9395.8 13.763332
     Kleine order (1 persoon / Snack)             90           907.7  1.329634
